In [38]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
import shap
import warnings
import logging
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler     
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.utils import check_array
from tensorflow.keras.losses import mean_squared_error
import seaborn as sns
from sklearn.utils.validation import check_is_fitted
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from mpl_toolkits.mplot3d import Axes3D
import tensorflow as tf
import math

from tensorflow.keras.layers import Input, Dense,Lambda
from tensorflow.python.keras import backend as K
from tensorflow.python.keras import layers as KL
from tensorflow.python.keras import models as KM

from sklearn.metrics import roc_auc_score
from scipy.optimize import curve_fit
from itertools import combinations
from keras.models import load_model
import random
from sklearn.datasets import make_swiss_roll
from sklearn.datasets import make_s_curve
from scipy.io import savemat
import time
import requests

In [39]:
disease_df_tmp = pd.read_csv('Data/Disease_hp.csv', encoding="gbk")
zscore_phentype_filter_final = pd.read_csv('Data/CamSu_zscore_phenotype.csv')

print(zscore_phentype_filter_final.shape[0])

1052


Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.


In [40]:
zscore_phentype_use = zscore_phentype_filter_final[~zscore_phentype_filter_final['mp_term_id'].str.contains('None', na=False)]

zscore_phentype_use = zscore_phentype_use.reset_index(drop=True)

print('Effective parameter numbers:',len(zscore_phentype_use.parameter_name.unique().tolist()))
print('Abnormal gene numbers:',len(zscore_phentype_use.gene_symbol.unique().tolist()))
print(zscore_phentype_use.shape[0])

Effective parameter numbers: 95
Abnormal gene numbers: 196
1052


In [41]:
IMPC_ODBAE_gene_df_tmp = zscore_phentype_use

In [42]:
IMPC_ODBAE_gene_df = IMPC_ODBAE_gene_df_tmp[~IMPC_ODBAE_gene_df_tmp['mp_term_id'].str.contains('MPATH', na=False)]
disease_df = disease_df_tmp[disease_df_tmp['database_id'].str.contains('OMIM', na=False)]
IMPC_ODBAE_gene_df = IMPC_ODBAE_gene_df.reset_index(drop=True)
disease_df = disease_df.reset_index(drop=True)

disease_list = disease_df.database_id.unique().tolist()
gene_list = IMPC_ODBAE_gene_df.gene_symbol.unique().tolist()
disease_count = disease_df.database_id.value_counts()
gene_count = IMPC_ODBAE_gene_df.gene_symbol.value_counts()

print(len(disease_list))
print(len(gene_list))
print(IMPC_ODBAE_gene_df.shape[0])

8326
196
1052


In [43]:
disease_df_index = disease_df.set_index('database_id')
gene_index = IMPC_ODBAE_gene_df.set_index('gene_symbol')

In [44]:
hp_list = list(disease_df_index['hpo_id'][disease_list[0]])
mp_list = list(gene_index['mp_term_id'][gene_list[0]])

print(hp_list)
print(mp_list)

['HP:0011097', 'HP:0002187', 'HP:0001518', 'HP:0032792', 'HP:0011451', 'HP:0010851', 'HP:0001789', 'HP:0200134', 'HP:0001522', 'HP:0000006', 'HP:0002643']
['MP:0001262', 'MP:0001262', 'MP:0001262']


In [45]:
url = "https://api-v3.monarchinitiative.org/v3/api/semsim/multicompare"
headers = {
    "accept": "application/json",
    "Content-Type": "application/json"
}

In [46]:
ob_dene = 'Pah-e'
ob_disease = 'OMIM:144250'

In [47]:
object_sets = []
tmp_dict = {}
tmp_dict['id'] = 'MGI'
tmp_dict['label'] = ob_dene
tmp_list = []
if gene_count.loc[ob_dene] == 1:
    tmp_df = gene_index.loc[ob_dene]
    tmp_list.append(tmp_df['mp_term_id'])
else:
    tmp_df = gene_index.loc[ob_dene]
    tmp_df = tmp_df.reset_index(drop=True)
    tmp_list = tmp_df.mp_term_id.unique().tolist()
tmp_dict['phenotypes'] = tmp_list
object_sets.append(tmp_dict)

In [48]:
print(object_sets)

[{'id': 'MGI', 'label': 'Pah-e', 'phenotypes': ['MP:0003921', 'MP:0005599', 'MP:0003394', 'MP:0010579', 'MP:0002953', 'MP:0001262', 'MP:0005291', 'MP:0005633', 'MP:0005627', 'MP:0003018', 'MP:0005565', 'MP:0005553', 'MP:0005568', 'MP:0002965', 'MP:0005344', 'MP:0000194', 'MP:0001566', 'MP:0002968', 'MP:0005178', 'MP:0001556', 'MP:0001552', 'MP:0005559', 'MP:0008806', 'MP:0011896', 'MP:0000182', 'MP:0005637', 'MP:0031059', 'MP:0010092', 'MP:0000592', 'MP:0001417']}]


In [49]:
data = {}
data['metric'] = "phenodigm_score"
subjects = []
tmp_match_df = pd.DataFrame()
disease_col = []
gene_col = []
ave_score = []
max_score = []
if disease_count.loc[ob_disease] == 1:
    tmp_df = disease_df_index.loc[ob_disease]
    subjects.append(tmp_df['hpo_id'])
else:
    tmp_df = disease_df_index.loc[ob_disease]
    tmp_df = tmp_df.reset_index(drop=True)
    subjects = tmp_df.hpo_id.unique().tolist()
data['subjects'] = subjects
print(subjects)
print(len(subjects))
print(tmp_list)
print(len(tmp_list)) 

['HP:0003124', 'HP:0003077', 'HP:0003141', 'HP:0003362', 'HP:0001658', 'HP:0001114', 'HP:0031798', 'HP:0000006', 'HP:0031800']
9
['MP:0003921', 'MP:0005599', 'MP:0003394', 'MP:0010579', 'MP:0002953', 'MP:0001262', 'MP:0005291', 'MP:0005633', 'MP:0005627', 'MP:0003018', 'MP:0005565', 'MP:0005553', 'MP:0005568', 'MP:0002965', 'MP:0005344', 'MP:0000194', 'MP:0001566', 'MP:0002968', 'MP:0005178', 'MP:0001556', 'MP:0001552', 'MP:0005559', 'MP:0008806', 'MP:0011896', 'MP:0000182', 'MP:0005637', 'MP:0031059', 'MP:0010092', 'MP:0000592', 'MP:0001417']
30


In [50]:
# 请求的 URL
url = "https://api-v3.monarchinitiative.org/v3/api/semsim/compare"

# 请求头
headers = {
    'accept': 'application/json',
    'Content-Type': 'application/json',
}

In [51]:
df = pd.DataFrame(columns = subjects, index = tmp_list)

for i in range(len(subjects)):
    for j in range(len(tmp_list)):
        print(i,j)
        data = {
            "subjects": [subjects[i]],
            "objects": [tmp_list[j]],
            "metric": "phenodigm_score"
        }
        response = requests.post(url, headers=headers, json=data)
        df[subjects[i]][tmp_list[j]] = response.json()['best_score']


0 0
0 1
0 2
0 3
0 4
0 5
0 6
0 7
0 8
0 9
0 10
0 11
0 12
0 13
0 14
0 15
0 16
0 17
0 18
0 19
0 20
0 21
0 22
0 23
0 24
0 25
0 26
0 27
0 28
0 29
1 0
1 1
1 2
1 3
1 4
1 5
1 6
1 7
1 8
1 9
1 10
1 11
1 12
1 13
1 14
1 15
1 16
1 17
1 18
1 19
1 20
1 21
1 22
1 23
1 24
1 25
1 26
1 27
1 28
1 29
2 0
2 1
2 2
2 3
2 4
2 5
2 6
2 7
2 8
2 9
2 10
2 11
2 12
2 13
2 14
2 15
2 16
2 17
2 18
2 19
2 20
2 21
2 22
2 23
2 24
2 25
2 26
2 27
2 28
2 29
3 0
3 1
3 2
3 3
3 4
3 5
3 6
3 7
3 8
3 9
3 10
3 11
3 12
3 13
3 14
3 15
3 16
3 17
3 18
3 19
3 20
3 21
3 22
3 23
3 24
3 25
3 26
3 27
3 28
3 29
4 0
4 1
4 2
4 3
4 4
4 5
4 6
4 7
4 8
4 9
4 10
4 11
4 12
4 13
4 14
4 15
4 16
4 17
4 18
4 19
4 20
4 21
4 22
4 23
4 24
4 25
4 26
4 27
4 28
4 29
5 0
5 1
5 2
5 3
5 4
5 5
5 6
5 7
5 8
5 9
5 10
5 11
5 12
5 13
5 14
5 15
5 16
5 17
5 18
5 19
5 20
5 21
5 22
5 23
5 24
5 25
5 26
5 27
5 28
5 29
6 0
6 1
6 2
6 3
6 4
6 5
6 6
6 7
6 8
6 9
6 10
6 11
6 12
6 13
6 14
6 15
6 16
6 17
6 18
6 19
6 20
6 21
6 22
6 23
6 24
6 25
6 26
6 27
6 28
6 29
7 0
7 1
7 2
7 3
7 4


In [52]:
print(df)

           HP:0003124 HP:0003077 HP:0003141 HP:0003362 HP:0001658 HP:0001114  \
MP:0003921   1.042783    1.09819    1.03434    1.03434   0.981898   1.124632   
MP:0005599   0.691446   0.736586     0.6847     0.6847   1.940319   0.686181   
MP:0003394   0.736586   0.791906   0.728447   0.728447   2.194423   0.671763   
MP:0010579   0.946767   0.987653   0.940434   0.940434    0.86904    1.03281   
MP:0002953   0.963177   1.006325   0.956511   0.956511   0.887731    1.03086   
MP:0001262   0.546523   0.583828   0.540974   0.540974   0.523799   0.513227   
MP:0005291   0.639928   0.697346   0.631671   0.631671   0.391465   0.241617   
MP:0005633   2.250754   2.213632   2.115049   2.115049    0.62315   0.722976   
MP:0005627   2.250754   2.213632   2.115049   2.115049    0.62315   0.722976   
MP:0003018   2.001056   2.058514   1.976502   1.976502   0.691636   0.767507   
MP:0005565   2.393092   2.325936   2.246816   2.246816   0.642328    0.73605   
MP:0005553   2.298144   2.225694   2.161